In [1]:
!pip install autogen_agentchat autogen_ext --quiet

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.80.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.


In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
import utils
import os

In [4]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_API_BASE'])

In [5]:
PLANNER_SYS_PROMPT = "You are a helpful assistant that can suggest a travel plan for a user based on their request."

planner_agent = AssistantAgent(
    "planner_agent",
    model_client=model_client,
    description="A helpful assistant that can plan trips.",
    system_message= PLANNER_SYS_PROMPT
)

In [6]:
LOCAL_SYS_PROMPT = """
You are a helpful assistant that can suggest authentic and interesting local activities or places
to visit for a user and can utilize any context information provided.
"""
local_agent = AssistantAgent(
    "local_agent",
    model_client=model_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message= LOCAL_SYS_PROMPT
)

In [7]:
LANG_SYS_PROMPT = """
"You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about
how best to address language or communication challenges for the given destination.

If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.
"""
language_agent = AssistantAgent(
    "language_agent",
    model_client=model_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message=LANG_SYS_PROMPT
)

In [8]:
TRAVEL_SYS_PROMPT = """
You are a helpful assistant that can take in all of the suggestions and advice from the other agents
and provide a detailed final travel plan.
You must ensure that the final plan is integrated and complete.

YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN.
When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.
"""
travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=model_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message=TRAVEL_SYS_PROMPT
)

In [9]:
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], termination_condition=termination
)
await Console(group_chat.run_stream(task="Plan a 3 day trip to Goa."))

await model_client.close()

---------- TextMessage (user) ----------
Plan a 3 day trip to Goa.
---------- TextMessage (planner_agent) ----------
Sure! Here’s a suggested itinerary for a 3-day trip to Goa, covering a mix of relaxation, sightseeing, and local culture.

### Day 1: Explore North Goa Beaches

**Morning:**
- **Arrival in Goa:** Aim to arrive in the morning. Check into your hotel or accommodation in North Goa (Areas like Calangute or Baga are great for beach lovers).
- **Breakfast:** Enjoy breakfast at a local café. Try a traditional Goan breakfast like Poha or upma.

**Midday:**
- **Visit Calangute Beach:** Relax and soak up the sun at Calangute Beach. You can engage in water activities like parasailing or jet skiing if you’re interested.

**Afternoon:**
- **Lunch:** Have lunch at a beach shack on Calangute Beach. Try some Goan seafood curry or vindaloo.
- **Explore Fort Aguada:** Visit this historical fort for a bit of history and stunning views of the Arabian Sea.

**Evening:**
- **Visit Sinquerim Be